# 00 · Galaxy10 DECaLS: data preparation & EDA  (issues #2, #3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ErenKerimoglu/CS464_Project/blob/feat/2-3-data-pipeline/notebooks/00_eda.ipynb)

**What this notebook does**
1. Clones the repo and installs the package.
2. Runs `scripts/prepare_data.py`: download (2.7 GB, MD5-verified) → dedup → frozen `split_v1` → train band stats → `images_256.npy` cache.
3. EDA: class counts, sample grids, pixel stats, **does the galaxy fit in the 180 px crop?**, redshift, near-duplicates.
4. Packs everything that goes into git (`data/splits/*`, `results/figures/eda/*`, `eda_summary.json`) into **`eda_outputs.zip`** for download.

**How to run:** *Runtime → Run all*. No GPU needed (CPU runtime is fine). It takes about 15–25 min, mostly the download.
Then upload `eda_outputs.zip` (or paste the summary printed at the end) so it can be committed on the branch.

In [ ]:
BRANCH = "feat/2-3-data-pipeline"   # change to "main" after the PR is merged
REPO = "https://github.com/ErenKerimoglu/CS464_Project.git"
DATA_DIR = "/content/data"           # outside the repo; ~6.5 GB
CROSSCHECK_NODUP = False             # True: also download astroNN's 2.7 GB de-duplicated file to cross-check our dedup

import os, subprocess, sys
if not os.path.exists("/content/CS464_Project"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/CS464_Project"], check=True)
%cd /content/CS464_Project
!git log --oneline -1
!pip install -q -e .
os.environ["GALAXY_DATA_DIR"] = DATA_DIR

## 1 · Download, deduplicate, split, band stats, cache

In [ ]:
!GALAXY_DATA_DIR={DATA_DIR} python scripts/prepare_data.py

## 2 · Load everything

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import importlib, galaxy.constants as C
importlib.reload(C)
from galaxy.data import eda
from galaxy.data.dedup import nearest_neighbor_sep_arcsec
from galaxy.data.download import load_metadata
from galaxy.data.split import split_summary

FIG = Path("results/figures/eda"); FIG.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

meta = load_metadata(C.RAW_DIR / C.DATASET_FILENAME)
split = pd.read_csv(C.SPLIT_FILE)
removed = pd.read_csv(C.REMOVED_FILE)
report = json.loads(C.DEDUP_REPORT_FILE.read_text())
stats = json.loads(C.BAND_STATS_FILE.read_text())
images = np.load(C.PROCESSED_IMAGES, mmap_mode="r")      # (17736, 256, 256, 3) uint8, rows = orig_index
labels = meta.label.to_numpy()
print(images.shape, len(split), "kept /", len(meta), "original")

## 3 · Class balance

In [ ]:
display(eda.class_count_table(labels))
assert tuple(np.bincount(labels, minlength=10)) == C.CLASS_COUNTS, "class counts differ from the documented ones"
save(eda.plot_class_counts(labels), "class_counts")

## 4 · Sample images per class (full 256 px and the 180 px crop the models see)

In [ ]:
save(eda.plot_samples_grid(images, labels, per_class=8, seed=0), "samples_grid")
save(eda.plot_samples_grid(images, labels, per_class=8, seed=0, crop=C.CROP_SIZE), "samples_grid_crop180")

## 5 · Pixel statistics per band (train split, 180 crop) — the frozen band stats

In [ ]:
rng = np.random.default_rng(0)
train_idx = split.loc[split.split == "train", "orig_index"].to_numpy()
sample_idx = np.sort(rng.choice(train_idx, min(2000, len(train_idx)), replace=False))
display(eda.pixel_stats(np.asarray(images[sample_idx])))
print("band_stats_v1.json:", stats["mean"], stats["std"])
fig, ax = plt.subplots(figsize=(7, 3))
sub = np.asarray(images[sample_idx[:300]])[:, 38:218, 38:218].reshape(-1, 3)
for b, (name, col) in enumerate(zip("grz", ["tab:green", "tab:red", "tab:purple"])):
    ax.hist(sub[:, b], bins=64, histtype="step", color=col, label=name, density=True, log=True)
ax.set_xlabel("pixel value (uint8)"); ax.legend(); ax.set_title("Pixel distributions (300 train images, crop)")
save(fig, "pixel_hist")

## 6 · Does the galaxy fit in the 180 px crop?  ← decides the crop before the split is frozen
Fraction of background-subtracted r-band flux inside the central 180×180 crop. Rotation-safe crops must be ≤ 181 px (PROJECT_PLAN §3.1).

**Caveat (result of the v1 run):** this whole-image metric is dominated by *neighbouring* sources and by positive
noise left after clipping in the outer ring (the ring is 51 % of the image area), so it is a lower bound, not the
size of the target galaxy. The worst cases below all show the central galaxy well inside the crop; use the
sample grids for the actual decision. A per-galaxy extent measure comes with the segmentation in #11.

In [ ]:
frac = np.concatenate([eda.crop_flux_fraction(np.asarray(images[s:s + 1000])) for s in range(0, len(images), 1000)])
meta["crop_flux_frac"] = frac
kept = meta.orig_index.isin(split.orig_index)
f = meta.loc[kept, "crop_flux_frac"]
print(f"median {f.median():.3f} | p5 {f.quantile(.05):.3f} | p1 {f.quantile(.01):.3f}")
print(f"< 0.90: {(f < .9).sum()} images ({(f < .9).mean()*100:.2f} %) | < 0.80: {(f < .8).sum()} ({(f < .8).mean()*100:.2f} %)")
display(meta[kept].groupby("label").crop_flux_frac.describe(percentiles=[.05, .5])[["mean", "5%", "50%", "min"]].round(3))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(f, bins=100, range=(0.5, 1.0), color="#4C72B0"); ax.axvline(0.9, color="k", ls="--", lw=1)
ax.set_xlabel("flux fraction inside 180 px crop (r band)"); ax.set_ylabel("images"); ax.set_yscale("log")
save(fig, "crop_flux_fraction")
worst = meta[kept].nsmallest(8, "crop_flux_frac")
fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, (_, r) in zip(axes, worst.iterrows()):
    ax.imshow(images[int(r.orig_index)]); ax.add_patch(plt.Rectangle((38, 38), 180, 180, fill=False, ec="y", lw=1))
    ax.set_title(f"{C.CLASS_SHORT[int(r.label)]} {r.crop_flux_frac:.2f}", fontsize=7); ax.axis("off")
save(fig, "crop_worst_cases")

## 7 · Redshift and pixel scale

In [ ]:
print("pxscale unique values:", np.unique(meta.pxscale.round(4))[:10])
fig, ax = plt.subplots(figsize=(8, 3))
ax.boxplot([meta.loc[(meta.label == c) & kept, "redshift"].dropna() for c in range(10)], showfliers=False)
ax.set_xticks(range(1, 11), C.CLASS_SHORT, rotation=30); ax.set_ylabel("redshift"); ax.set_title("Redshift per class")
save(fig, "redshift_by_class")

## 8 · Near-duplicates (#3): nearest-neighbour separations, sensitivity, examples

In [ ]:
nn = nearest_neighbor_sep_arcsec(meta.ra.values, meta.dec.values)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.log10(np.clip(nn, 1e-3, None)), bins=120, color="#4C72B0")
ax.axvline(np.log10(C.DEDUP_RADIUS_ARCSEC), color="r", ls="--", label='1" radius'); ax.legend()
ax.set_xlabel("log10 nearest-neighbour separation [arcsec]"); ax.set_ylabel("galaxies"); ax.set_yscale("log")
save(fig, "nn_separation")
display(pd.DataFrame(report["sensitivity"]))
print("removed:", removed.reason.value_counts().to_dict())
dups = removed[removed.reason == "duplicate"].head(8)
pairs = list(zip(dups.kept_index, dups.orig_index))
conf = removed[removed.reason == "label_conflict"].groupby("group_id").orig_index.apply(list).head(4)
pairs += [tuple(g[:2]) for g in conf]
if pairs:
    title = f"Removed: {len(dups)} same-label duplicates (top), {len(conf)} label-conflict pairs (bottom)"
    save(eda.plot_pairs(images, pairs, labels, title), "duplicate_pairs")

In [ ]:
if CROSSCHECK_NODUP:
    from galaxy.data.download import download_file
    from galaxy.data.dedup import find_pairs
    nd_path = download_file(C.NODUP_URL, C.RAW_DIR / C.NODUP_FILENAME)
    nd = load_metadata(nd_path)
    both = pd.concat([split[["ra", "dec"]], nd[["ra", "dec"]]], ignore_index=True)
    print(f"astroNN NoDuplicated: {len(nd)} images | ours: {len(split)} kept")
    # how many of our kept galaxies have a counterpart in astroNN's file (within 1")
    pr = find_pairs(both.ra.values, both.dec.values, 1.0)
    cross = pr[(pr[:, 0] < len(split)) & (pr[:, 1] >= len(split))]
    print(f"our kept galaxies matched in astroNN's file: {len(np.unique(cross[:, 0]))} / {len(split)}")

## 9 · The frozen split

In [ ]:
summ = split_summary(split)
display(summ)
dev = max((summ[f"{s} %"] - summ["all %"]).abs().max() for s in ("train", "val", "test"))
print(f"max class-share deviation: {dev:.3f} pp (acceptance: < 0.5)")
print(split.split.value_counts().to_dict())

## 10 · Summary + outputs to commit

In [ ]:
summary = {
    "n_original": len(meta), "n_kept": len(split), "removed": removed.reason.value_counts().to_dict(),
    "split_counts": split.split.value_counts().to_dict(), "max_class_share_deviation_pp": round(float(dev), 3),
    "band_stats": {"mean": stats["mean"], "std": stats["std"]},
    "crop180_flux_fraction": {"median": round(float(f.median()), 4), "p5": round(float(f.quantile(.05)), 4),
                              "p1": round(float(f.quantile(.01)), 4), "n_below_0.9": int((f < .9).sum()),
                              "n_below_0.8": int((f < .8).sum())},
    "dedup_sensitivity": report["sensitivity"],
}
Path("results/figures/eda/eda_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
!zip -q -r /content/eda_outputs.zip data/splits results/figures/eda
try:
    from google.colab import files; files.download("/content/eda_outputs.zip")
except ImportError:
    print("Not on Colab: outputs are in data/splits and results/figures/eda")